In [73]:
import sys, numpy as np

from keras.datasets import mnist
from spacy.lang.ja.syntax_iterators import labels
from srsly.msgpack import epoch
from torch import layout

np.random.seed(1)

# Подготовка данных для тренировки и тестирования
(x_train, y_train), (x_test, y_test) = mnist.load_data()


def prepare_images(images, size=1000):
    return images[0:size].reshape(size, 28 * 28) / 255


def prepare_labels(labels, size=1000):
    labels = labels[0:size]
    on_hot_labels = np.zeros((len(labels), 10))

    for i, l in enumerate(labels):
        on_hot_labels[i][l] = 1

    return on_hot_labels


images, labels = prepare_images(x_train), prepare_labels(y_train)
test_images, test_labels = prepare_images(x_test), prepare_labels(y_test)

In [ ]:
relu = lambda x: (x >= 0) * x
relu2deriv = lambda x: (x >= 0)

alpha = 0.005
epochs = 350
hidden_size = 40
pixels = 28 * 28
num_labels = 10  # 0..9

weights_0_1 = 0.2 * np.random.random((pixels, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for epoch in range(epochs):
    error, correct_cnt = (0.0, 0)

    for i in range(len(images)):
        layer_0 = images[i:i + 1]
        layer_1 = relu(layer_0 @ weights_0_1)
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = layer_1 @ weights_1_2

        error += np.sum((labels[i:i + 1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i + 1]))

        layer_2_delta = (labels[i:i + 1] - layer_2)
        layer_1_delta = (layer_2_delta @ weights_1_2.T) * relu2deriv(layer_1)

        weights_1_2 += alpha * (layer_1.T @ layer_2_delta)
        weights_0_1 += alpha * (layer_0.T @ layer_1_delta)

    print(
        "Epoch:" + str(epoch) +
        " Error:" + str(error / float(len(images)))[0:5] +
        " Correct:" + str(correct_cnt / float(len(images)))
    )

In [42]:
# Тестируем на самой себе (из любопытсва)
error, correct_cnt = (0.0, 0)

for i in range(len(images)):
    layer_0 = images[i:i + 1]
    layer_1 = relu(layer_0 @ weights_0_1)
    layer_2 = layer_1 @ weights_1_2

    error += np.sum((labels[i:i + 1] - layer_2) ** 2)
    correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i + 1]))

print(
    "Error:" + str(error / float(len(images)))[0:5] +
    " Correct:" + str(correct_cnt / float(len(images)))
)

Error:0.379 Correct:0.833


In [43]:
# Тестируем на тестовой выборке
error, correct_cnt = (0.0, 0)

for i in range(len(test_images)):
    layer_0 = test_images[i:i + 1]
    layer_1 = relu(layer_0 @ weights_0_1)
    layer_2 = layer_1 @ weights_1_2

    error += np.sum((test_labels[i:i + 1] - layer_2) ** 2)
    correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i + 1]))

print(
    "Error:" + str(error / float(len(test_images)))[0:5] +
    " Correct:" + str(correct_cnt / float(len(test_images)))
)

Error:0.487 Correct:0.733


In [61]:
# Пакетный градиентный спуск. Этот метод увеличивает скорость обучения и улучшает сходимость
# но я как-то не заметил... было 23 секунды, стало 33. Шта?
import numpy as np

np.random.seed(1)


def relu(x):
    c += 1
    return (x >= 0) * x  # Возвращает x, если x > 0


def relu2deriv(output):
    return output >= 0  # Возвращает 1, если output >0


batch_size = 100
alpha, iterations = (0.001, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)
weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))

        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = np.dot(layer_1, weights_1_2)

        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k + 1]) == np.argmax(labels[batch_start + k:batch_start + k + 1]))

            layer_2_delta = (labels[batch_start:batch_end] - layer_2) / batch_size
            layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
            layer_1_delta *= dropout_mask

            weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
            weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if (j % 10 == 0):
        test_error = 0.0
        test_correct_cnt = 0

        for i in range(len(test_images)):
            layer_0 = test_images[i:i + 1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)

            test_error += np.sum((test_labels[i:i + 1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i + 1]))

        print("I:" + str(j) +
              " Test-Err:" + str(test_error / float(len(test_images)))[0:5] +
              " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) +
              " Train-Err:" + str(error / float(len(images)))[0:5] +
              " Train-Acc:" + str(correct_cnt / float(len(images)))
              )

I:0 Test-Err:0.830 Test-Acc:0.348 Train-Err:1.284 Train-Acc:0.165
I:10 Test-Err:0.590 Test-Acc:0.679 Train-Err:0.591 Train-Acc:0.672
I:20 Test-Err:0.533 Test-Acc:0.726 Train-Err:0.532 Train-Acc:0.729
I:30 Test-Err:0.506 Test-Acc:0.753 Train-Err:0.498 Train-Acc:0.754
I:40 Test-Err:0.488 Test-Acc:0.762 Train-Err:0.489 Train-Acc:0.749
I:50 Test-Err:0.476 Test-Acc:0.766 Train-Err:0.468 Train-Acc:0.775
I:60 Test-Err:0.469 Test-Acc:0.773 Train-Err:0.452 Train-Acc:0.799
I:70 Test-Err:0.462 Test-Acc:0.78 Train-Err:0.453 Train-Acc:0.792
I:80 Test-Err:0.468 Test-Acc:0.772 Train-Err:0.457 Train-Acc:0.786
I:90 Test-Err:0.463 Test-Acc:0.776 Train-Err:0.454 Train-Acc:0.799
I:100 Test-Err:0.465 Test-Acc:0.782 Train-Err:0.447 Train-Acc:0.796
I:110 Test-Err:0.459 Test-Acc:0.775 Train-Err:0.426 Train-Acc:0.816
I:120 Test-Err:0.461 Test-Acc:0.783 Train-Err:0.431 Train-Acc:0.813
I:130 Test-Err:0.460 Test-Acc:0.785 Train-Err:0.434 Train-Acc:0.816
I:140 Test-Err:0.464 Test-Acc:0.785 Train-Err:0.437 Train-Ac

33000

In [78]:

# activation functions.
def tanh(x):
    return np.tanh(x)


def tanh2deriv(output):
    return 1 - (output ** 2)


def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)


alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

W_1 = (0.02 * np.random.random((pixels_per_image, hidden_size))) - 0.01
W_2 = (0.2 * np.random.random((hidden_size, num_labels))) - 0.1

# Training Loop
for j in range(iterations):  # epoches
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):  # batches
        batch_start, batch_end = (i * batch_size), ((i + 1) * batch_size)

        # Forward Propagation
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0, W_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1, W_2))

        # benchmarking
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k + 1]) == np.argmax(labels[batch_start + k:batch_start + k + 1]))

        # backpropagation
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(W_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        # optimization
        W_2 += alpha * layer_1.T.dot(layer_2_delta)
        W_1 += alpha * layer_0.T.dot(layer_1_delta)

    test_correct_cnt = 0

    for i in range(len(test_images)):  # test images
        # predict
        layer_0 = test_images[i:i + 1]
        layer_1 = tanh(np.dot(layer_0, W_1))
        layer_2 = np.dot(layer_1, W_2)

        # benchmark
        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i + 1]))

    if (j % 10 == 0):
        print(
            f"I: {j} | Test-Acc: {round(test_correct_cnt / float(len(test_images)), 5)} | Train-Acc: {round(correct_cnt / float(len(images)), 5)}")

I: 0 | Test-Acc: 0.4012 | Train-Acc: 0.163
I: 10 | Test-Acc: 0.6704 | Train-Acc: 0.702
I: 20 | Test-Acc: 0.6915 | Train-Acc: 0.721
I: 30 | Test-Acc: 0.7287 | Train-Acc: 0.744
I: 40 | Test-Acc: 0.7646 | Train-Acc: 0.786
I: 50 | Test-Acc: 0.7897 | Train-Acc: 0.82
I: 60 | Test-Acc: 0.807 | Train-Acc: 0.838
I: 70 | Test-Acc: 0.8176 | Train-Acc: 0.864
I: 80 | Test-Acc: 0.8275 | Train-Acc: 0.869
I: 90 | Test-Acc: 0.8359 | Train-Acc: 0.871
I: 100 | Test-Acc: 0.841 | Train-Acc: 0.887
I: 110 | Test-Acc: 0.8444 | Train-Acc: 0.897
I: 120 | Test-Acc: 0.8458 | Train-Acc: 0.905
I: 130 | Test-Acc: 0.8478 | Train-Acc: 0.901
I: 140 | Test-Acc: 0.8517 | Train-Acc: 0.904
I: 150 | Test-Acc: 0.8538 | Train-Acc: 0.913
I: 160 | Test-Acc: 0.8564 | Train-Acc: 0.917
I: 170 | Test-Acc: 0.8579 | Train-Acc: 0.922
I: 180 | Test-Acc: 0.8595 | Train-Acc: 0.92
I: 190 | Test-Acc: 0.8605 | Train-Acc: 0.921
I: 200 | Test-Acc: 0.8627 | Train-Acc: 0.925
I: 210 | Test-Acc: 0.8632 | Train-Acc: 0.928
I: 220 | Test-Acc: 0.8647